In [1]:
!python3 -m pip install scikit-learn joblib pandas


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# =======================
# 1. LOAD DATA
# =======================
df = pd.read_csv("data/weatherAUS.csv")

df["RainTomorrow"] = df["RainTomorrow"].map({"Yes": 1, "No": 0})
df = df.dropna(subset=["RainTomorrow"])

X = df.drop(columns=["RainTomorrow"])
y = df["RainTomorrow"]

# =======================
# 2. DEFINE COLUMNS
# =======================
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

# =======================
# 3. FIT IMPUTERS
# =======================
num_medians = X[num_cols].median()
cat_modes = X[cat_cols].mode().iloc[0]

X[num_cols] = X[num_cols].fillna(num_medians)
X[cat_cols] = X[cat_cols].fillna(cat_modes)

# =======================
# 4. FIT ONE-HOT ENCODER
# =======================
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)
encoder.fit(X[cat_cols])

cat_ohe = encoder.transform(X[cat_cols])
cat_ohe_df = pd.DataFrame(cat_ohe, columns=encoder.get_feature_names_out(cat_cols))

# =======================
# 5. BUILD FINAL TRAINING DATAFRAME
# =======================
X_final = pd.concat([X[num_cols].reset_index(drop=True), cat_ohe_df], axis=1)

dummy_columns = X_final.columns.tolist()

# =======================
# 6. TRAIN RANDOM FOREST
# =======================
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

model.fit(X_final, y)

# =======================
# 7. SAVE PREPROCESSOR + MODEL
# =======================
preprocessor_data = {
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "num_medians": num_medians,
    "cat_modes": cat_modes,
    "encoder": encoder,
    "dummy_columns": dummy_columns
}

joblib.dump(preprocessor_data, "model/preprocessor.joblib", compress=("lzma", 3))
m = joblib.load("model/aussie_rain.joblib")
joblib.dump(m, "model/aussie_rain.joblib", compress=("lzma", 9))

print("DONE — saved compatible model and preprocessor!")


DONE — saved compatible model and preprocessor!


In [4]:
ls -lh model/


total 164472
-rw-r--r--@ 1 oleksandraandriyishyn  staff    76M  7 гру 17:30 aussie_rain.joblib
-rw-r--r--@ 1 oleksandraandriyishyn  staff   5,6K  7 гру 17:27 preprocessor.joblib
